In [1]:
import asyncio, sys
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv(dotenv_path='../.env')
assert os.getenv('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing in .env'
print('Environment loaded ✓')

Environment loaded ✓


In [3]:
llm = LLM(
    model='openrouter/openai/gpt-oss-120b:free',
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv('OPENROUTER_API_KEY'),
)
print('LLM configured ✓')

LLM configured ✓


In [4]:
SAMPLE_ANALYSIS = """
1. Main topic / product feature: Wireless headphones
2. Specific praises:
   - Decent sound for the price
3. Specific complaints:
   - Muddy bass at high volumes
   - Battery life only 6 hours vs 10 advertised
   - Plasticky build quality
4. Emotional tone: Frustrated, leaning disappointed
5. Factual claims: Battery life of 6 hours vs 10 hours advertised
"""
print(SAMPLE_ANALYSIS)


1. Main topic / product feature: Wireless headphones
2. Specific praises:
   - Decent sound for the price
3. Specific complaints:
   - Muddy bass at high volumes
   - Battery life only 6 hours vs 10 advertised
   - Plasticky build quality
4. Emotional tone: Frustrated, leaning disappointed
5. Factual claims: Battery life of 6 hours vs 10 hours advertised



In [5]:
classifier = Agent(
    role='Sentiment Classifier',
    goal=(
        "Read the analyst's report and commit to a final sentiment "
        'label: POSITIVE, NEGATIVE, or NEUTRAL.'
    ),
    backstory=(
        'A precise classification specialist. You always pick exactly '
        'one of three labels and back it with a confidence score.'
    ),
    llm=llm,
    verbose=True,
)

classify_task = Task(
    description=(
        'Based on the analysis report below, output the final verdict in '
        'EXACTLY this format:\n'
        'Sentiment: <POSITIVE|NEGATIVE|NEUTRAL>\n'
        'Confidence: <number between 0.0 and 1.0>\n'
        'Justification: <one short sentence>\n\n'
        'ANALYSIS REPORT:\n{analysis_report}'
    ),
    expected_output='Three lines: Sentiment, Confidence, Justification.',
    agent=classifier,
)

crew = Crew(
    agents=[classifier],
    tasks=[classify_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff(inputs={'analysis_report': SAMPLE_ANALYSIS})

print('\n=========== CLASSIFIER OUTPUT ===========')
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cc499177-89bc-43aa-a344-828fe5a05cdf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on the analysis report below, output the final verdict in EXACTLY this format:                     │
│  Sentiment: <POSITIVE|NEGATIVE|NEUTRAL>                                                                         │
│  Confidence: <number between 0.0 and 1.0>                                                                       │
│  Justification: <one short sentence>                                                                            │
│                                                                                                                 │
│  ANALYSIS REPORT:                                                                                               │
│                                                                                                                 │
│  1. Main topic / product feature: Wireless headphones                                                           │
│  2. Specific praises:                                                                                           │
│     - Decent sound for the price                                                                                │
│  3. Specific complaints:                                                                                        │
│     - Muddy bass at high volumes                                                                                │
│     - Battery life only 6 hours vs 10 advertised                                                                │
│     - Plasticky build quality                                                                                   │
│  4. Emotional tone: Frustrated, leaning disappointed                                                            │
│  5. Factual claims: Battery life of 6 hours vs 10 hours advertised                                              │
│                                                                                                                 │
│  ID: 0be2d875-f5f4-4d0c-8033-3d25cc5ade58                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sentiment Classifier                                                                                    │
│                                                                                                                 │
│  Task: Based on the analysis report below, output the final verdict in EXACTLY this format:                     │
│  Sentiment: <POSITIVE|NEGATIVE|NEUTRAL>                                                                         │
│  Confidence: <number between 0.0 and 1.0>                                                                       │
│  Justification: <one short sentence>                                                                            │
│                                                                                                                 │
│  ANALYSIS REPORT:                                                                                               │
│                                                                                                                 │
│  1. Main topic / product feature: Wireless headphones                                                           │
│  2. Specific praises:                                                                                           │
│     - Decent sound for the price                                                                                │
│  3. Specific complaints:                                                                                        │
│     - Muddy bass at high volumes                                                                                │
│     - Battery life only 6 hours vs 10 advertised                                                                │
│     - Plasticky build quality                                                                                   │
│  4. Emotional tone: Frustrated, leaning disappointed                                                            │
│  5. Factual claims: Battery life of 6 hours vs 10 hours advertised                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sentiment Classifier                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Sentiment: NEGATIVE                                                                                            │
│  Confidence: 0.86                                                                                               │
│  Justification: The reviewer’s frustration over muddy bass, short battery life, and cheap build outweighs the   │
│  modest praise for sound quality.                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the analysis report below, output the final verdict in EXACTLY this format:                     │
│  Sentiment: <POSITIVE|NEGATIVE|NEUTRAL>                                                                         │
│  Confidence: <number between 0.0 and 1.0>                                                                       │
│  Justification: <one short sentence>                                                                            │
│                                                                                                                 │
│  ANALYSIS REPORT:                                                                                               │
│                                                                                                                 │
│  1. Main topic / product feature: Wireless headphones                                                           │
│  2. Specific praises:                                                                                           │
│     - Decent sound for the price                                                                                │
│  3. Specific complaints:                                                                                        │
│     - Muddy bass at high volumes                                                                                │
│     - Battery life only 6 hours vs 10 advertised                                                                │
│     - Plasticky build quality                                                                                   │
│  4. Emotional tone: Frustrated, leaning disappointed                                                            │
│  5. Factual claims: Battery life of 6 hours vs 10 hours advertised                                              │
│                                                                                                                 │
│  Agent: Sentiment Classifier                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cc499177-89bc-43aa-a344-828fe5a05cdf                                                                       │
│  Final Output: Sentiment: NEGATIVE                                                                              │
│  Confidence: 0.86                                                                                               │
│  Justification: The reviewer’s frustration over muddy bass, short battery life, and cheap build outweighs the   │
│  modest praise for sound quality.                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=========== CLASSIFIER OUTPUT ===========
Sentiment: NEGATIVE  
Confidence: 0.86  
Justification: The reviewer’s frustration over muddy bass, short battery life, and cheap build outweighs the modest praise for sound quality.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯